## ANN Classifier

In [1]:
import torch
from typing import Tuple, Dict, Any, List
import torch.nn.functional as F
import numpy as np
import time
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class ANNClassifier:
    def __init__(self, input_dim=None, hidden_dim1=224, hidden_dim2=128, C=1.0, max_iter=100, verbose=True, random_state=42):
        self.C = C
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.max_iter = max_iter
        self.random_state = random_state
        self.verbose = verbose
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim = input_dim  # Will be set during fit() if None
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.model = None

    def compute_loss(self, preds, labels):
        loss = self.loss_func(preds, labels)
        wreg = 0.5 * sum((param.norm(p=2) for param in self.model.parameters()))  # L2 regularization
        return loss.mean() + (1.0 / self.C) * wreg

    def predict_proba(self, feats):
        feats = feats.to(self.device)
        with torch.no_grad():
            return torch.softmax(self.model(feats), dim=1)

    def fit(self, feats, labels):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        # Set input_dim if not set
        if self.input_dim is None:
            self.input_dim = feats.shape[1]

        self.model = torch.nn.Sequential(
            torch.nn.Linear(self.input_dim, self.hidden_dim1),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim1, self.hidden_dim2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim2, 9)  # Assuming 9 classes
        ).to(self.device)

        for layer in self.model:
            if isinstance(layer, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(layer.weight)

        feats = feats.to(self.device)
        labels = labels.long().to(self.device)

        opt = torch.optim.Adam(self.model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)

        min_loss = np.inf
        patience = 10
        patience_counter = 0

        for epoch in range(self.max_iter):
            opt.zero_grad()
            preds = self.model(feats)
            loss = self.compute_loss(preds, labels)
            loss.backward()
            opt.step()
            scheduler.step(loss)

            if self.verbose and epoch % 10 == 0:
                print(f"Epoch {epoch}: Loss: {loss:.4f}")

            if loss < min_loss:
                min_loss = loss
                patience_counter = 0
                best_model = self.model.state_dict()
            else:
                patience_counter += 1

            # if patience_counter >= patience:
            #     print(f"Early stopping at epoch {epoch}")
            #     break

        self.model.load_state_dict(best_model)

        if self.verbose:
            print(f"Final Loss: {min_loss:.4f}")

In [2]:
def test_ANN_probe(
    classifier: ANNClassifier,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    prefix: str = "ann_",
    verbose: bool = True,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    if verbose:
        print(f"ANN Probe Evaluation: Test shape {test_feats.shape}")

    probs_all = classifier.predict_proba(test_feats).cpu().numpy()
    preds_all = np.argmax(probs_all, axis=1)
    targets_all = test_labels.cpu().numpy()
    roc_kwargs = {"multi_class": "ovo", "average": "macro"}

    # ⬇️ Calculate accuracy
    accuracy = np.mean(preds_all == targets_all)
    print(f"✅ Test Accuracy: {accuracy * 100:.2f}%")

    # ⬇️ Eval metrics
    eval_metrics = get_eval_metrics(targets_all, preds_all, probs_all, True, prefix, roc_kwargs)
    dump = {"preds_all": preds_all, "probs_all": probs_all, "targets_all": targets_all}

    # ⬇️ Confusion matrix
    conf_matrix = confusion_matrix(targets_all, preds_all)
    print("Confusion Matrix:")
    plot_confusion_matrix(conf_matrix)
    
    # ⬇️ New plots for deeper insight
    plot_classification_report(targets_all, preds_all)
    plot_roc_multiclass(targets_all, probs_all, n_classes=9)
    plot_prediction_confidence(probs_all, preds_all, targets_all)


    return eval_metrics, dump


In [3]:
# Do this in main
import pandas as pd

df= pd.read_csv(r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv")

def is_non_white_patch(patch_path: str, valid_patch_names: set) -> int:
    patch_name = os.path.basename(patch_path)
    return int(patch_name in valid_patch_names)

def extract_coords(patch_name: str):
    try:
        base = os.path.splitext(patch_name)[0]
        parts = base.split('_')
        x = int([p for p in parts if p.startswith('x')][0][1:])
        y = int([p for p in parts if p.startswith('y')][0][1:])
        return x, y
    except Exception as e:
        print(f"❌ Error extracting coords from {patch_name}: {e}")
        return None, None

## Conch 224 five crop patch and crc100k 224 size

In [4]:
import json

with open(r'D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\best_parameters\best_params_conch.json', 'r') as f:
    best_params = json.load(f)

print("✅ Loaded best_params from best_params_uni2.json",best_params)

✅ Loaded best_params from best_params_uni2.json {'hidden_dim1': 224, 'hidden_dim2': 128, 'C': 10.0, 'max_iter': 500}


### conch features 224 five crop pipeline

In [5]:
model_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\models\ann_crc100k_conch15.pth"

In [6]:
input_dim = 768

# Initialize classifier
model = ANNClassifier(
    input_dim=input_dim,
    hidden_dim1=best_params['hidden_dim1'],
    hidden_dim2=best_params['hidden_dim2'],
    C=best_params['C'],
    max_iter=best_params.get('max_iter', 100),
    verbose=True
)

# Build model architecture (required before loading weights)
model.model = torch.nn.Sequential(
    torch.nn.Linear(model.input_dim, model.hidden_dim1),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(model.hidden_dim1, model.hidden_dim2),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.3),
    torch.nn.Linear(model.hidden_dim2, 9)  # 9 classes assumed
).to(model.device)

# Load trained weights
model.model = torch.load(model_path, map_location=model.device)
model.model.to(model.device)
model.model.eval()

Sequential(
  (0): Linear(in_features=768, out_features=224, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=224, out_features=128, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=128, out_features=9, bias=True)
)

## New appraoch label with batches

In [7]:
import os
import h5py
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# ========== CONFIGURATION ==========
patches_root = r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data"
features_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5_fivecrop_Updated"
non_white_csv = r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv"
BATCH_SIZE = 64  # You can adjust this based on your GPU memory

patch_groups_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\Patch_groups_dict\connch_224_fivecrop"
output_slide_features = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\2-Averaging_FiveCrop\conch_224_fivecrop"

In [10]:
import pandas as pd

# Load valid patch names
non_white_df = pd.read_csv(non_white_csv)

valid_patch_names = set(
    non_white_df['patch_name'].astype(str).str.replace('.png', '.pt', regex=False)
)

# Repeat each row 5 times
df_expanded = non_white_df.loc[non_white_df.index.repeat(5)].copy()

# Add updated_patchName column
df_expanded["updated_patchName"] = (
    df_expanded["patch_name"].str.replace(".png", "", regex=False)
    + "_" + (df_expanded.groupby(level=0).cumcount() + 1).astype(str) + ".png"
)

df_expanded

,patch_location,patch_name,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,label,updated_patchName
0,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...
0,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...
0,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...
0,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...
0,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377932,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_1.png
1377932,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_2.png
1377932,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_3.png
1377932,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_4.png


## Final Pipeline

In [11]:
import os
import torch
from tqdm import tqdm

# Collect all updates here for later update
updates = []
count = 0

for slide_name in tqdm(os.listdir(features_root), desc="🔍 Processing slides"):
    slide_feature_folder = os.path.join(features_root, slide_name)
    count += 1

    print(f"{count}: Processing slide: {slide_name}")

    if not os.path.isdir(slide_feature_folder):
        continue

    pt_files = sorted([
        f for f in os.listdir(slide_feature_folder)
        if f.endswith(".pt")
    ])

    batch_features = []
    batch_patch_names = []

    for idx, pt_file in enumerate(pt_files):
        pt_path = os.path.join(slide_feature_folder, pt_file)

        if is_non_white_patch(pt_path, valid_patch_names) == 0:
            continue

        try:
            features = torch.load(pt_path, map_location='cpu')  # (5, 768)
            if not isinstance(features, torch.Tensor):
                raise ValueError(f"Expected tensor, got {type(features)}")

            if features.shape != (5, 768):
                print(f"⚠️ Unexpected shape in {pt_file}: {features.shape}")
                continue

            # Directly extend the batch features with slices from the tensor
            batch_features.extend([features[i] for i in range(5)])

            # Generate corresponding patch names
            base = os.path.splitext(pt_file)[0]
            batch_patch_names.extend([f"{base}_{i+1}.png" for i in range(5)])

        except Exception as e:
            print(f"❌ Error reading {pt_path}: {e}")
            continue

        # Run model if batch full or last patch
        last_patch = (idx == len(pt_files) - 1)
        if len(batch_features) >= BATCH_SIZE or last_patch:
            try:
                tensor_batch = torch.stack(batch_features).to(model.device)  # shape: (B, 768)

                model.model.eval()
                with torch.no_grad():
                    outputs = model.model(tensor_batch)
                    predictions = torch.argmax(outputs, dim=1).cpu().tolist()

                updates.extend(zip(batch_patch_names, predictions))

            except Exception as e:
                print(f"❌ Inference error: {e}")

            # Free memory
            del tensor_batch, outputs
            torch.cuda.empty_cache()

            # Reset batch
            batch_features.clear()
            batch_patch_names.clear()

🔍 Processing slides:   0%|                                                                    | 0/417 [00:00<?, ?it/s]

1: Processing slide: TCGA-3L-AA1B_nonMSIH


🔍 Processing slides:   0%|▏                                                           | 1/417 [00:00<05:36,  1.24it/s]

2: Processing slide: TCGA-4N-A93T_nonMSIH


🔍 Processing slides:   0%|▎                                                           | 2/417 [00:02<08:48,  1.27s/it]

...

🔍 Processing slides: 100%|███████████████████████████████████████████████████████▋| 415/417 [2:55:19<01:11, 35.81s/it]

416: Processing slide: TCGA-T9-A92H_nonMSIH


🔍 Processing slides: 100%|███████████████████████████████████████████████████████▊| 416/417 [2:56:00<00:37, 37.13s/it]

417: Processing slide: TCGA-WS-AB45_MSIH


🔍 Processing slides: 100%|████████████████████████████████████████████████████████| 417/417 [2:56:40<00:00, 25.42s/it]


## Post Processing

In [12]:
update_df = pd.DataFrame(updates, columns=['updated_patchName', 'tissue_label'])
update_df

,updated_patchName,tissue_label
0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,0
1,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,0
2,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,0
3,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,0
4,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,0
...,...,...
6889660,TCGA-WS-AB45_MSIH_x9664_y37888_patch00199_1.png,4
6889661,TCGA-WS-AB45_MSIH_x9664_y37888_patch00199_2.png,4
6889662,TCGA-WS-AB45_MSIH_x9664_y37888_patch00199_3.png,4
6889663,TCGA-WS-AB45_MSIH_x9664_y37888_patch00199_4.png,2


In [13]:
len(set(update_df["updated_patchName"]))

6889665

In [14]:
# update_df = pd.DataFrame(updates, columns=['updated_patchName', 'tissue_label'])
df_expanded.set_index('updated_patchName', inplace=True)
df_expanded.update(update_df.set_index('updated_patchName'))
df_expanded.reset_index(inplace=True)

In [15]:
df_expanded = df_expanded.merge(update_df, on='updated_patchName', how='inner')

# Save the updated CSV if needed
df_expanded.to_csv("conch_labeled_df.csv", index=False)

In [16]:
df_expanded

,updated_patchName,patch_location,patch_name,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,label,tissue_label
0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
1,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
2,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
3,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
4,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.png,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6889660,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_1.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,7
6889661,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_2.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,1
6889662,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_3.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,3
6889663,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_4.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.png,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,1


## Slide Aggregation

In [17]:
non_white_df= df_expanded.copy()

In [18]:
# non_white_df['patch_name'] = non_white_df['patch_name'].str.replace('.png', '.pt', regex=False)
import re
non_white_df['patch_name'] = non_white_df['patch_name'].str.replace('.png', '.pt', regex=False)
# non_white_df['patch_name'] = non_white_df['patch_name'].str.replace(r'_patch\d{5}', '', regex=True)

In [19]:
non_white_df

,updated_patchName,patch_location,patch_name,patch_number,patch_x,patch_y,avg_R,avg_G,avg_B,std_R,std_G,std_B,label,tissue_label
0,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.pt,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
1,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.pt,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
2,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.pt,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
3,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.pt,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
4,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159_...,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-3L-AA1B_nonMSIH_x10288_y12512_patch00159.pt,159,10288,12512,237.72,225.94,230.83,15.18,32.27,25.42,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6889660,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_1.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.pt,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,7
6889661,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_2.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.pt,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,1
6889662,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_3.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.pt,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,3
6889663,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531_4.png,D:/Aamir Gulzar/KSA_project2/dataset/patch_dat...,TCGA-WS-AB45_MSIH_x54208_y25088_patch04531.pt,4531,54208,25088,231.00,221.49,227.52,26.98,38.33,28.96,0,1


## Slide aggregation pipeline

In [20]:
import os
import torch
import numpy as np
from tqdm import tqdm
import pandas as pd

# ---------------- CONFIG ----------------
root_dir = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\Conch_v1_5_fivecrop_Updated"
saving_path = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\2-Averaging_FiveCrop\conch_224_fivecrop"
num_classes = 9
# ----------------------------------------

os.makedirs(saving_path, exist_ok=True)

# Load label CSV once. Expect columns 'patch_name' and 'tissue_label'.
# We'll create two maps:
#   1) split_name_map: exact filename without extension -> label  (for cases non_white has _1.._5)
#   2) base_name_map: base patch name (without _1.._5) -> label (fallback)
non_white_df['patch_name'] = non_white_df['patch_name'].astype(str)

split_name_map = {}  # exact match
base_name_map = {}   # fallback by base (without last _idx)
for pn, label in zip(non_white_df['patch_name'], non_white_df['tissue_label']):
    name = os.path.splitext(pn)[0]
    split_name_map[name] = label
    # derive base by removing trailing _<num> if present
    parts = name.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        base = parts[0]
    else:
        base = name
    # only set if not set, keep first occurrence
    if base not in base_name_map:
        base_name_map[base] = label

# Process each slide folder
for slide_name in tqdm(os.listdir(root_dir), desc="Slides"):
    slide_path = os.path.join(root_dir, slide_name)
    if not os.path.isdir(slide_path):
        continue

    files = [f for f in os.listdir(slide_path) if f.endswith('.pt')]
    if not files:
        continue

    # We'll collect features and labels in two lists, then convert to numpy arrays
    features_list = []
    labels_list = []

    for fname in files:
        pt_path = os.path.join(slide_path, fname)
        # load to CPU
        try:
            t = torch.load(pt_path, map_location="cpu")
        except Exception as e:
            print(f"ERROR loading {pt_path}: {e}")
            continue

        # normalize tensor to 2D: (num_crops, D)
        if isinstance(t, torch.Tensor):
            t = t.detach().cpu()
            t = t.squeeze()
            if t.ndim == 1:
                crops = t.unsqueeze(0).numpy()  # (1, D)
            elif t.ndim == 2:
                crops = t.numpy()               # (N, D) e.g., (5, 768)
            else:
                # flatten any extra dims into second dim (rare)
                crops = t.view(t.shape[0], -1).numpy()
        else:
            # skip non-tensor objects
            print(f"Skipping (not tensor): {pt_path}")
            continue

        base_name = os.path.splitext(fname)[0]  # original base without extension

        # For each crop of this patch, create split name and determine label (fast, without saving)
        # We avoid any heavy string ops in loops; simple concatenation and map lookups only.
        for idx in range(crops.shape[0]):
            split_name = f"{base_name}_{idx+1}"  # like file_1, file_2, ...
            features_list.append(crops[idx])     # append numpy 1D array
            # label resolution: exact split name -> fallback to base map
            lbl = split_name_map.get(split_name)
            if lbl is None:
                # fallback: try base name (original patch name without _idx)
                # base_candidate: remove trailing _<digit> if any
                parts = base_name.rsplit('_', 1)
                if len(parts) == 2 and parts[1].isdigit():
                    base_candidate = parts[0]
                else:
                    base_candidate = base_name
                lbl = base_name_map.get(base_candidate, np.nan)
            labels_list.append(lbl)

    # If nothing collected skip
    if not features_list:
        continue

    # Convert lists to arrays
    feats = np.stack(features_list, axis=0).astype(np.float32)  # shape (N, D)
    labels = np.array(labels_list, dtype=np.float32)            # shape (N,)

    # Compute feature dimension
    D = feats.shape[1]

    # Compute group averages per class (0..num_classes-1)
    group_avgs = np.zeros((num_classes, D), dtype=np.float32)

    # Use vectorized grouping: for each label, mask and mean
    # Slight optimization: if labels array contains NaNs, comparisons will be False
    for cls in range(num_classes):
        mask = (labels == cls)
        if mask.any():
            group_avgs[cls] = feats[mask].mean(axis=0)
        else:
            # remains zeros
            pass

    final_tensor = torch.from_numpy(group_avgs)  # shape (num_classes, D)

    # Save the aggregated slide-level tensor
    save_path = os.path.join(saving_path, f"{slide_name}.pt")
    torch.save(final_tensor, save_path)

Slides: 100%|██████████████████████████████████████████████████████████████████████| 417/417 [4:08:01<00:00, 35.69s/it]


In [23]:
import torch

# Path to your .pt file
pt_path = r"D:\Aamir Gulzar\KSA_project2\dataset\slide_aggregation\2-Averaging_FiveCrop\TCGA-AA-3678_nonMSIH.pt"

# Load to CPU to avoid GPU dependency
tensor = torch.load(pt_path, map_location="cpu")

# If it's a tensor
if isinstance(tensor, torch.Tensor):
    print("Shape:", tensor.shape)
    print("Values:\n", tensor)
else:
    print("Loaded object type:", type(tensor))
    print("Value:", tensor)


Shape: torch.Size([9, 768])
Values:
 tensor([[-0.5847, -0.3429, -1.2125,  ...,  2.1690,  0.7562,  0.4149],
        [-0.3209, -0.4401, -0.8908,  ...,  2.2065,  0.4518,  0.4030],
        [-0.4376, -0.6947, -1.2316,  ...,  1.7534,  0.4005,  0.4160],
        ...,
        [-0.4609, -0.2849, -1.4860,  ...,  1.7434,  1.0410,  0.4686],
        [-0.5748, -0.4138, -1.4791,  ...,  1.8163,  0.5344,  0.4690],
        [-0.4650, -0.6612, -1.4538,  ...,  1.6533,  0.6672,  0.4750]])
